# NHL Data Cleaning

This notebook cleans and preprocesses the raw NHL play-by-play data.

## Steps:
1. Filter for shot events only
2. Flatten nested JSON columns
3. Remove unnecessary columns
4. Standardize coordinates
5. Handle missing data
6. Create target variable (is_goal)


In [1]:
import pandas as pd
import numpy as np
import sys

# Add src to path
sys.path.append('../../')
from src.data.processors.data_cleaning import (
    filter_shot_events,
    flatten_nested_columns,
    remove_unnecessary_columns,
    standardize_coordinates,
    handle_missing_data,
    create_goal_target
)


## Load Raw Data


In [2]:
# Load raw data
INPUT_FILE = "../../data/raw/nhl_raw_plays.parquet"
df = pd.read_parquet(INPUT_FILE)

print(f"Original shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


Original shape: (413676, 11)
Columns: ['eventId', 'periodDescriptor', 'timeInPeriod', 'timeRemaining', 'situationCode', 'homeTeamDefendingSide', 'typeCode', 'typeDescKey', 'sortOrder', 'details', 'pptReplayUrl']


,eventId,periodDescriptor,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,details,pptReplayUrl
0,102,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:00,20:00,1551,left,520,period-start,8,None,None
1,101,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:00,20:00,1551,left,502,faceoff,9,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
2,8,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:35,19:25,1551,left,516,stoppage,15,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
3,103,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:35,19:25,1551,left,502,faceoff,17,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
4,9,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",00:48,19:12,1551,left,503,hit,20,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None


## Step 1: Filter for Shot Events


In [3]:
# Identify all unique events
print("Unique event types:")
print(df["typeDescKey"].unique())

# Filter for shot events
shot_events = ['shot-on-goal', 'missed-shot', 'blocked-shot', 'goal']
df_shots = filter_shot_events(df, shot_events)

print(f"\nOriginal shape: {df.shape}")
print(f"After filtering shots: {df_shots.shape}")
df_shots.head()


Unique event types:
['period-start' 'faceoff' 'stoppage' 'hit' 'giveaway' 'shot-on-goal'
 'takeaway' 'missed-shot' 'blocked-shot' 'goal' 'penalty'
 'delayed-penalty' 'period-end' 'game-end' 'shootout-complete'
 'failed-shot-attempt']

Original shape: (413676, 11)
After filtering shots: (160123, 11)


,eventId,periodDescriptor,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,details,pptReplayUrl
6,63,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",01:01,18:59,1551,left,506,shot-on-goal,22,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
7,151,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",01:10,18:50,1551,left,506,shot-on-goal,23,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
9,70,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",01:47,18:13,1551,left,506,shot-on-goal,31,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
16,152,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",02:57,17:03,1551,left,506,shot-on-goal,48,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None
18,95,"{'maxRegulationPeriods': 3, 'number': 1, 'peri...",03:51,16:09,1551,left,507,missed-shot,60,"{'assist1PlayerId': None, 'assist1PlayerTotal'...",None


## Step 2: Flatten Nested Columns


In [4]:
# Flatten nested JSON columns
df_shots_clean = flatten_nested_columns(df_shots)

print(f"Shape after flattening: {df_shots_clean.shape}")
print(f"\nColumns ({len(df_shots_clean.columns)}):")
for col in df_shots_clean.columns:
    print(f"  - {col}")
df_shots_clean.head()


Removed duplicate columns. New shape: (160123, 48)
Shape after flattening: (160123, 48)

Columns (48):
  - eventId
  - timeInPeriod
  - timeRemaining
  - situationCode
  - homeTeamDefendingSide
  - typeCode
  - typeDescKey
  - sortOrder
  - pptReplayUrl
  - assist1PlayerId
  - assist1PlayerTotal
  - assist2PlayerId
  - assist2PlayerTotal
  - awaySOG
  - awayScore
  - blockingPlayerId
  - committedByPlayerId
  - descKey
  - discreteClip
  - discreteClipFr
  - drawnByPlayerId
  - duration
  - eventOwnerTeamId
  - goalieInNetId
  - highlightClip
  - highlightClipFr
  - highlightClipSharingUrl
  - highlightClipSharingUrlFr
  - hitteePlayerId
  - hittingPlayerId
  - homeSOG
  - homeScore
  - losingPlayerId
  - playerId
  - reason
  - scoringPlayerId
  - scoringPlayerTotal
  - secondaryReason
  - servedByPlayerId
  - shootingPlayerId
  - shotType
  - winningPlayerId
  - xCoord
  - yCoord
  - zoneCode
  - maxRegulationPeriods
  - period
  - periodType


,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,pptReplayUrl,assist1PlayerId,...,servedByPlayerId,shootingPlayerId,shotType,winningPlayerId,xCoord,yCoord,zoneCode,maxRegulationPeriods,period,periodType
0,63,01:01,18:59,1551,left,506,shot-on-goal,22,None,NaN,...,None,8478178.0,wrist,None,58.0,-25.0,O,3,1,REG
1,151,01:10,18:50,1551,left,506,shot-on-goal,23,None,NaN,...,None,8478010.0,tip-in,None,81.0,8.0,O,3,1,REG
2,70,01:47,18:13,1551,left,506,shot-on-goal,31,None,NaN,...,None,8479661.0,snap,None,55.0,30.0,O,3,1,REG
3,152,02:57,17:03,1551,left,506,shot-on-goal,48,None,NaN,...,None,8479591.0,wrist,None,58.0,-30.0,O,3,1,REG
4,95,03:51,16:09,1551,left,507,missed-shot,60,None,NaN,...,None,8476887.0,wrist,None,-63.0,33.0,O,3,1,REG


## Step 3: Remove Unnecessary Columns


In [5]:
# Remove columns not relevant for shot analysis
df_shots_clean = remove_unnecessary_columns(df_shots_clean)

print(f"Shape after removing columns: {df_shots_clean.shape}")
df_shots_clean.head()


Shape after removing columns: (160123, 28)


,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,assist1PlayerId,assist1PlayerTotal,...,homeScore,scoringPlayerId,scoringPlayerTotal,shootingPlayerId,shotType,xCoord,yCoord,zoneCode,period,periodType
0,63,01:01,18:59,1551,left,506,shot-on-goal,22,NaN,NaN,...,NaN,NaN,NaN,8478178.0,wrist,58.0,-25.0,O,1,REG
1,151,01:10,18:50,1551,left,506,shot-on-goal,23,NaN,NaN,...,NaN,NaN,NaN,8478010.0,tip-in,81.0,8.0,O,1,REG
2,70,01:47,18:13,1551,left,506,shot-on-goal,31,NaN,NaN,...,NaN,NaN,NaN,8479661.0,snap,55.0,30.0,O,1,REG
3,152,02:57,17:03,1551,left,506,shot-on-goal,48,NaN,NaN,...,NaN,NaN,NaN,8479591.0,wrist,58.0,-30.0,O,1,REG
4,95,03:51,16:09,1551,left,507,missed-shot,60,NaN,NaN,...,NaN,NaN,NaN,8476887.0,wrist,-63.0,33.0,O,1,REG


## Step 4: Create Target Variable


In [6]:
# Create is_goal column
df_shots_clean = create_goal_target(df_shots_clean)

print("Goal Counts (1=Goal, 0=No Goal):")
print(df_shots_clean["is_goal"].value_counts())
print(f"\nGoal Rate: {df_shots_clean['is_goal'].mean():.2%}")


Goal Counts (1=Goal, 0=No Goal):
is_goal
0    151855
1      8268
Name: count, dtype: int64

Goal Rate: 5.16%


## Step 5: Standardize Coordinates


In [7]:
# Standardize coordinates so all shots are aimed at positive-x net
df_shots_clean = standardize_coordinates(df_shots_clean)

print("Coordinates standardized. All shots are now aimed at the positive-x net.")
print("\nCoordinate ranges:")
print(f"  xCoord: {df_shots_clean['xCoord'].min():.1f} to {df_shots_clean['xCoord'].max():.1f}")
print(f"  yCoord: {df_shots_clean['yCoord'].min():.1f} to {df_shots_clean['yCoord'].max():.1f}")


Coordinates standardized. All shots are now aimed at the positive-x net.

Coordinate ranges:
  xCoord: 0.0 to 99.0
  yCoord: -42.0 to 42.0


## Step 6: Handle Missing Data


In [8]:
# Check missing data before handling
print("Missing values before handling:")
missing_before = df_shots_clean.isnull().sum()
print(missing_before[missing_before > 0])

# Handle missing data
df_shots_clean = handle_missing_data(df_shots_clean)

# Check missing data after handling
print("\nMissing values after handling:")
missing_after = df_shots_clean.isnull().sum()
print(missing_after[missing_after > 0])

# Check shot type distribution
print("\nShot Type Distribution:")
print(df_shots_clean['shotType'].value_counts())


Missing values before handling:
assist1PlayerId       152563
assist1PlayerTotal    152563
assist2PlayerId       154005
assist2PlayerTotal    154005
awaySOG                88408
awayScore             151855
blockingPlayerId      115454
goalieInNetId          45457
homeSOG                88408
homeScore             151855
scoringPlayerId       151855
scoringPlayerTotal    152037
shootingPlayerId        8268
shotType               44691
dtype: int64

Missing values after handling:
awaySOG                88408
awayScore             151855
blockingPlayerId      115454
goalieInNetId          45457
homeSOG                88408
homeScore             151855
scoringPlayerId       151855
scoringPlayerTotal    152037
shootingPlayerId        8268
dtype: int64

Shot Type Distribution:
shotType
wrist           63244
unknown         44691
snap            16062
slap            13554
tip-in          10102
backhand         8284
deflected        2327
wrap-around       907
poke              434
bat        

## Step 7: Save Cleaned Data


In [9]:
# Save cleaned data
# First, check for and remove any duplicate columns (PyArrow doesn't allow duplicates)
if df_shots_clean.columns.duplicated().any():
    duplicates = df_shots_clean.columns[df_shots_clean.columns.duplicated()].unique()
    print(f"Warning: Found duplicate columns before saving: {list(duplicates)}")
    df_shots_clean = df_shots_clean.loc[:, ~df_shots_clean.columns.duplicated()]
    print(f"Removed duplicates. New shape: {df_shots_clean.shape}")

# Fix data types for PyArrow compatibility
# Convert object columns with mixed types to strings
print("\nFixing data types for PyArrow compatibility...")
for col in df_shots_clean.columns:
    if df_shots_clean[col].dtype == 'object':
        # Check if column has mixed types (numbers and strings)
        try:
            # Try to convert to numeric - if it fails, it's a string column
            pd.to_numeric(df_shots_clean[col], errors='raise')
        except (ValueError, TypeError):
            # It's a string column - ensure all values are strings
            df_shots_clean[col] = df_shots_clean[col].astype(str)
            # Replace 'nan' strings with empty string or 'None'
            df_shots_clean[col] = df_shots_clean[col].replace('nan', 'None')
            df_shots_clean[col] = df_shots_clean[col].replace('NaN', 'None')
            df_shots_clean[col] = df_shots_clean[col].replace('<NA>', 'None')

print("Data types fixed!")

OUTPUT_FILE = "../../data/processed/cleaned_shots.parquet"
df_shots_clean.to_parquet(OUTPUT_FILE, index=False)

print(f"Cleaned data saved to {OUTPUT_FILE}")
print(f"\nFinal DataFrame shape: {df_shots_clean.shape}")
print(f"Total shots: {len(df_shots_clean):,}")
print(f"Total goals: {df_shots_clean['is_goal'].sum():,}")
print(f"Goal rate: {df_shots_clean['is_goal'].mean():.2%}")

df_shots_clean.head()



Fixing data types for PyArrow compatibility...
Data types fixed!
Cleaned data saved to ../../data/processed/cleaned_shots.parquet

Final DataFrame shape: (160123, 29)
Total shots: 160,123
Total goals: 8,268
Goal rate: 5.16%


,eventId,timeInPeriod,timeRemaining,situationCode,homeTeamDefendingSide,typeCode,typeDescKey,sortOrder,assist1PlayerId,assist1PlayerTotal,...,scoringPlayerId,scoringPlayerTotal,shootingPlayerId,shotType,xCoord,yCoord,zoneCode,period,periodType,is_goal
0,63,01:01,18:59,1551,left,506,shot-on-goal,22,None,None,...,NaN,NaN,8478178.0,wrist,58.0,-25.0,O,1,REG,0
1,151,01:10,18:50,1551,left,506,shot-on-goal,23,None,None,...,NaN,NaN,8478010.0,tip-in,81.0,8.0,O,1,REG,0
2,70,01:47,18:13,1551,left,506,shot-on-goal,31,None,None,...,NaN,NaN,8479661.0,snap,55.0,30.0,O,1,REG,0
3,152,02:57,17:03,1551,left,506,shot-on-goal,48,None,None,...,NaN,NaN,8479591.0,wrist,58.0,-30.0,O,1,REG,0
4,95,03:51,16:09,1551,left,507,missed-shot,60,None,None,...,NaN,NaN,8476887.0,wrist,63.0,-33.0,O,1,REG,0
